In [1]:
import requests
import time
import pandas as pd
from datetime import datetime, timedelta

In [2]:
API_KEY = "qrZUHabbbGHJWU95326BITrpe1ZX6SbC79MvFmbIKuEICM9l"

MAX_MINUTE_RETRIES = 3   # retry attempts for minute limit
MINUTE_SLEEP = 60        # NYT allows ~5 requests per minute

def fetch_articles(query, begin_date, end_date, page):

    url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

    params = {
        "api-key": API_KEY,
        "q": query,
        "begin_date": begin_date,
        "end_date": end_date,
        "page": page,
    }

    retries = 0

    while retries <= MAX_MINUTE_RETRIES:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            return data.get("response", {}).get("docs", [])

        if response.status_code == 429:
            retries += 1

            if retries > MAX_MINUTE_RETRIES:
                print("Daily rate limit likely reached.")
                return None  # signal to stop everything

            print(f"Minute rate limit hit. Sleeping {MINUTE_SLEEP}s...")
            time.sleep(MINUTE_SLEEP)
            continue
    
        print("Error:", response.status_code)
        return []

    return None


def get_all_articles(query, start_date, end_date):

    all_articles = []
    current_date = start_date
    stop_collection = False

    while current_date <= end_date and not stop_collection:

        month_end = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
        if month_end > end_date:
            month_end = end_date

        begin_str = current_date.strftime("%Y%m%d")
        end_str = month_end.strftime("%Y%m%d")

        print(f"Fetching {begin_str} to {end_str}")

        month_articles = []          # store articles for THIS month only
        month_failed = False         # track if any request failed

        for page in range(5):

            docs = fetch_articles(query, begin_str, end_str, page)

            if docs is None:
                month_failed = True  # mark month as incomplete
                stop_collection = True
                break

            if not docs:
                break

            month_articles.extend(docs)
            
            time.sleep(12)

        # Only commit the month if ALL requests succeeded
        if not month_failed:
            all_articles.extend(month_articles)
        else:
            print(f"Skipping incomplete month {begin_str}-{end_str}")

        current_date = month_end + timedelta(days=1)

    print(f"Returning {len(all_articles)} articles collected so far.")
    return pd.json_normalize(all_articles)

In [3]:
# Run it
start = datetime(2006, 1, 1)
end = datetime(2026, 2, 28)

df = get_all_articles("economy", start, end)

df.head()

Fetching 20060101 to 20060131
Fetching 20060201 to 20060228
Fetching 20060301 to 20060331
Fetching 20060401 to 20060430
Fetching 20060501 to 20060531
Fetching 20060601 to 20060630
Fetching 20060701 to 20060731
Fetching 20060801 to 20060831
Fetching 20060901 to 20060930
Fetching 20061001 to 20061031
Fetching 20061101 to 20061130
Fetching 20061201 to 20061231
Fetching 20070101 to 20070131
Fetching 20070201 to 20070228
Fetching 20070301 to 20070331
Fetching 20070401 to 20070430
Fetching 20070501 to 20070531
Fetching 20070601 to 20070630
Fetching 20070701 to 20070731
Fetching 20070801 to 20070831
Fetching 20070901 to 20070930
Fetching 20071001 to 20071031
Fetching 20071101 to 20071130
Fetching 20071201 to 20071231
Fetching 20080101 to 20080131
Fetching 20080201 to 20080229
Fetching 20080301 to 20080331
Fetching 20080401 to 20080430
Fetching 20080501 to 20080531
Fetching 20080601 to 20080630
Fetching 20080701 to 20080731
Fetching 20080801 to 20080831
Fetching 20080901 to 20080930
Fetching 2

,abstract,document_type,_id,keywords,news_desk,print_page,print_section,pub_date,section_name,snippet,...,headline.kicker,headline.print_headline,multimedia.caption,multimedia.credit,multimedia.default.url,multimedia.default.height,multimedia.default.width,multimedia.thumbnail.url,multimedia.thumbnail.height,multimedia.thumbnail.width
0,"The United States generated about 200,000 new ...",article,nyt://article/313ed1c5-243e-55ef-b28b-73dc8a67...,"[{'name': 'Subject', 'value': 'United States E...",Business/Financial Desk,00001,C,2006-01-07T05:00:00Z,Business,"The United States generated about 200,000 new ...",...,,Bush Cites 2 Million New Jobs in 2005 and Heal...,,,,0,0,,0,0
1,"The Chinese statistics, showing a national eco...",article,nyt://article/2905419b-8fde-5d18-ad0e-3faf04a2...,"[{'name': 'Location', 'value': 'China', 'rank'...",Business / World Business,,,2006-01-25T05:00:00Z,Business,"The Chinese statistics, showing a national eco...",...,,Chinese Economy Becomes 4th Largest in the World,,,,0,0,,0,0
2,Economic growth weakened unexpectedly in the f...,article,nyt://article/4bf669cc-a2f0-5b46-8880-b52fdba3...,"[{'name': 'Subject', 'value': 'United States E...",Business/Financial Desk,00001,C,2006-01-28T05:00:00Z,Business,Economic growth weakened unexpectedly in the f...,...,,U.S. Economy Slowed Sharply at End of 2005,,,,0,0,,0,0
3,"The Chinese statistics, showing a national eco...",article,nyt://article/101dc4c1-0d6d-5620-916a-749fcc35...,"[{'name': 'Location', 'value': 'China', 'rank'...",Business / World Business,,,2006-01-25T05:00:00Z,Business,"The Chinese statistics, showing a national eco...",...,,Chinese Economy Grows to 4th Largest in the World,,,,0,0,,0,0
4,Government statistics show China's economy gre...,article,nyt://article/fa267891-747c-5a32-8c29-ca2b6787...,"[{'name': 'Location', 'value': 'China', 'rank'...",Business/Financial Desk,00010,C,2006-01-25T05:00:00Z,Business,Government statistics show China's economy gre...,...,,Economy in China Is No. 4 in World,,,,0,0,,0,0


In [4]:
df.to_csv("nytimes_economy_1.csv", index=False)

In [5]:
df_2006 = pd.read_csv("nytimes_economy_1.csv")

In [7]:
df_2006.shape

(5000, 28)

In [8]:
df.shape

(5000, 28)

In [10]:
df_2006.isna().sum()

abstract                        527
document_type                     0
_id                               0
keywords                          0
news_desk                       783
print_page                     1938
print_section                  1934
pub_date                          0
section_name                      4
snippet                         531
source                            4
subsection_name                1640
type_of_material                163
uri                               0
web_url                           0
word_count                        0
byline.original                 503
headline.main                     0
headline.kicker                2967
headline.print_headline        1049
multimedia.caption             3698
multimedia.credit              3140
multimedia.default.url         4026
multimedia.default.height         0
multimedia.default.width          0
multimedia.thumbnail.url       3602
multimedia.thumbnail.height       0
multimedia.thumbnail.width  